<a href="https://colab.research.google.com/github/leoleocb/TrabajoParcial_UPC/blob/main/Sistema_Inventario_Facturacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# CONEXION A GOOGLE DRIVE
# ==========================================

from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

carpeta = "/content/drive/MyDrive/ProyectoInventario"

os.makedirs(carpeta, exist_ok=True)

ruta_drive = carpeta + "/inventario.db"
ruta_db = "/content/inventario.db"

if os.path.exists(ruta_drive):
    shutil.copy(ruta_drive, ruta_db)
    print("Base de datos recuperada correctamente.")
else:
    print("Se creara una nueva base de datos.")

print("Carpeta del proyecto lista.")

Mounted at /content/drive
Se creara una nueva base de datos.
Carpeta del proyecto lista.


In [ ]:

# ==========================================
# CREAR BASE DE DATOS SQLITE
# ==========================================

import sqlite3
import shutil

conexion = sqlite3.connect(ruta_db)
cursor = conexion.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS productos (
    codigo TEXT PRIMARY KEY,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL,
    stock INTEGER NOT NULL,
    stock_minimo INTEGER NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS ventas (
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha TEXT NOT NULL,
    total REAL NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS detalle_ventas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    venta_id INTEGER NOT NULL,
    codigo_producto TEXT NOT NULL,
    cantidad INTEGER NOT NULL,
    precio_unitario REAL NOT NULL,
    subtotal REAL NOT NULL,
    FOREIGN KEY (venta_id) REFERENCES ventas(id_venta),
    FOREIGN KEY (codigo_producto) REFERENCES productos(codigo)
)
""")

productos_prueba = [
    ("P001", "Teclado Logitech", 85.00, 20, 5),
    ("P002", "Mouse Gamer", 45.00, 15, 5),
    ("P003", "Monitor Samsung", 650.00, 4, 5),
    ("P004", "Audifonos Sony", 120.00, 12, 4),
    ("P005", "Memoria USB 32GB", 35.00, 30, 10),
    ("P006", "Disco SSD 500GB", 220.00, 3, 5),
    ("P007", "Laptop Lenovo", 2500.00, 8, 2),
    ("P008", "Cable HDMI", 25.00, 40, 10),
    ("P009", "Webcam Logitech", 180.00, 2, 5),
    ("P010", "Parlante Bluetooth", 95.00, 18, 5)
]

cursor.executemany("""
INSERT OR IGNORE INTO productos
(codigo, nombre, precio, stock, stock_minimo)
VALUES (?, ?, ?, ?, ?)
""", productos_prueba)

cursor.execute("SELECT COUNT(*) FROM ventas")
cantidad_ventas = cursor.fetchone()[0]

if cantidad_ventas == 0:
    ventas_prueba = [
        ("2026-10-01", 120.50),
        ("2026-10-01", 45.00),
        ("2026-10-02", 650.00),
        ("2026-10-02", 85.00),
        ("2026-10-02", 130.00),
        ("2026-10-03", 2500.00),
        ("2026-10-04", 35.00),
        ("2026-10-04", 45.00),
        ("2026-10-05", 220.00),
        ("2026-10-05", 95.00),
        ("2026-10-06", 180.00),
        ("2026-10-06", 120.00),
        ("2026-10-07", 300.00)
    ]

    cursor.executemany("""
    INSERT INTO ventas (fecha, total)
    VALUES (?, ?)
    """, ventas_prueba)

conexion.commit()

cursor.execute("SELECT COUNT(*) FROM productos")
total_productos = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM ventas")
total_ventas = cursor.fetchone()[0]

print("===================================")
print("     BASE DE DATOS SQLITE")
print("===================================")
print("Productos registrados:", total_productos)
print("Ventas registradas:", total_ventas)

conexion.close()

shutil.copy(ruta_db, ruta_drive)

print("\nBase de datos guardada correctamente.")


     BASE DE DATOS SQLITE
Productos registrados: 10
Ventas registradas: 13

Base de datos guardada correctamente.


In [ ]:

# ============================================================
# SISTEMA DE MONITOREO DE INVENTARIO Y FACTURACION
# CURSO: FUNDAMENTOS DE PROGRAMACION
# ============================================================

import sqlite3
import shutil
import pandas as pd


# ============================================================
# FUNCION PARA GUARDAR CAMBIOS EN GOOGLE DRIVE
# ============================================================

def guardar_cambios():
    shutil.copy(ruta_db, ruta_drive)


# ============================================================
# MODULO 1: INVENTARIO
# ============================================================

# Registrar nuevo producto
def registrar_producto():

    print("\n--- REGISTRO DE PRODUCTOS ---")

    codigo = input("Ingrese codigo: ").strip()

    if codigo == "":
        print("Error: El codigo no puede estar vacio.")
        return

    conexion = sqlite3.connect(ruta_db)
    cursor = conexion.cursor()

    # Verificar codigo existente
    cursor.execute(
        "SELECT * FROM productos WHERE codigo = ?",
        (codigo,)
    )

    if cursor.fetchone() is not None:
        print("Error: El codigo ya existe.")
        conexion.close()
        return

    nombre = input("Ingrese nombre: ").strip()

    if nombre == "":
        print("Error: El nombre no puede estar vacio.")
        conexion.close()
        return

    try:
        precio = float(input("Ingrese precio: S/ "))
        stock = int(input("Ingrese stock inicial: "))
        stock_minimo = int(input("Ingrese stock minimo: "))

    except ValueError:
        print("Error: Ingrese valores numericos.")
        conexion.close()
        return

    if precio < 0 or stock < 0 or stock_minimo < 0:
        print("Error: No se permiten valores negativos.")
        conexion.close()
        return

    # Registrar producto en SQLite
    cursor.execute("""
    INSERT INTO productos
    (codigo, nombre, precio, stock, stock_minimo)
    VALUES (?, ?, ?, ?, ?)
    """, (codigo, nombre, precio, stock, stock_minimo))

    conexion.commit()
    conexion.close()

    guardar_cambios()

    print("\nProducto registrado correctamente.")


# Mostrar todos los productos
def mostrar_productos():

    conexion = sqlite3.connect(ruta_db)
    cursor = conexion.cursor()

    cursor.execute("SELECT * FROM productos ORDER BY codigo")
    productos = cursor.fetchall()

    print("\n--- LISTA DE PRODUCTOS ---")

    if len(productos) == 0:
        print("No hay productos registrados.")

    else:
        for producto in productos:
            print("--------------------------------")
            print("Codigo:", producto[0])
            print("Nombre:", producto[1])
            print("Precio: S/", producto[2])
            print("Stock actual:", producto[3])
            print("Stock minimo:", producto[4])

        print("--------------------------------")
        print("Total de productos:", len(productos))

    conexion.close()


# Buscar producto
def buscar_producto():

    print("\n--- BUSCAR PRODUCTO ---")

    codigo = input("Ingrese codigo: ")

    conexion = sqlite3.connect(ruta_db)
    cursor = conexion.cursor()

    cursor.execute(
        "SELECT * FROM productos WHERE codigo = ?",
        (codigo,)
    )

    producto = cursor.fetchone()

    if producto is not None:
        print("\nProducto encontrado:")
        print("Codigo:", producto[0])
        print("Nombre:", producto[1])
        print("Precio: S/", producto[2])
        print("Stock:", producto[3])
        print("Stock minimo:", producto[4])

    else:
        print("Producto no encontrado.")

    conexion.close()


# Actualizar stock


def actualizar_stock():

    print("\n--- ACTUALIZAR STOCK ---")

    codigo = input("Ingrese codigo: ").strip().upper()

    conexion = None

    try:
        conexion = sqlite3.connect(ruta_db, timeout=15)
        cursor = conexion.cursor()

        cursor.execute(
            "SELECT * FROM productos WHERE codigo = ?",
            (codigo,)
        )

        producto = cursor.fetchone()

        if producto is None:
            print("Producto no encontrado.")
            return

        print("Producto:", producto[1])
        print("Stock actual:", producto[3])

        try:
            nuevo_stock = int(input("Ingrese nuevo stock: "))
        except ValueError:
            print("Error: Ingrese un numero entero.")
            return

        if nuevo_stock < 0:
            print("Error: El stock no puede ser negativo.")
            return

        cursor.execute(
            "UPDATE productos SET stock = ? WHERE codigo = ?",
            (nuevo_stock, codigo)
        )

        conexion.commit()

        print("Stock actualizado correctamente.")

    except sqlite3.Error as error:
        if conexion is not None:
            conexion.rollback()
        print("Error en la base de datos:", error)

    finally:
        if conexion is not None:
            conexion.close()

    if conexion is not None:
        guardar_cambios()



# Menu de inventario
def modulo_inventario():

    while True:

        print("\n========================================")
        print("          MODULO DE INVENTARIO")
        print("========================================")
        print("  [1] Registrar nuevo producto")
        print("  [2] Mostrar productos registrados")
        print("  [3] Buscar producto")
        print("  [4] Actualizar stock")
        print("  [5] Volver al menu principal")
        print("========================================")

        opcion = input("Ingrese una opcion (1-5): ")

        if opcion == "1":
            registrar_producto()

        elif opcion == "2":
            mostrar_productos()

        elif opcion == "3":
            buscar_producto()

        elif opcion == "4":
            actualizar_stock()

        elif opcion == "5":
            print("Regresando al menu principal...")
            break

        else:
            print("Opcion no valida.")


# ============================================================
# MODULO 2: FACTURACION
# RESPONSABLE: INTEGRANTE 2
# ============================================================

# ============================================================
# MODULO 2: FACTURACION (NUEVA VENTA)
# RESPONSABLE: INTEGRANTE 2
# ============================================================
def modulo_facturacion():
    while True:
        print("\n========================================")
        print("          MODULO DE FACTURACION")
        print("========================================")
        print("  [1] Realizar nueva venta")
        print("  [2] Volver al menu principal")
        print("========================================")
        opcion = input("Ingrese una opcion (1-2): ").strip()

        # Conectar con la base de datos SQLite
        conexion = sqlite3.connect(ruta_db)
        cursor = conexion.cursor()

        if opcion == "1":
            print("\n--- NUEVA VENTA ---")
            codigo = input("Ingrese el codigo del producto: ").strip()

            # Buscar el producto
            cursor.execute("SELECT codigo, nombre, precio, stock FROM productos WHERE codigo = ?", (codigo,))
            producto = cursor.fetchone()

            if producto is None:
                print("Error: El producto no existe.")
                conexion.close()
                continue

            codigo_prod, nombre_prod, precio_prod, stock_actual = producto
            print(f"Producto: {nombre_prod}")
            print(f"Precio Unitario: S/ {precio_prod:.2f}")
            print(f"Stock Disponible: {stock_actual}")

            try:
                cantidad = int(input("Ingrese la cantidad a vender: "))
            except ValueError:
                print("Error: Debe ingresar un numero entero.")
                conexion.close()
                continue

            if cantidad <= 0:
                print("Error: La cantidad debe ser mayor a cero.")
                conexion.close()
                continue

            if cantidad > stock_actual:
                print(f"Error: Stock insuficiente. Solo hay {stock_actual} unidades disponibles.")
                conexion.close()
                continue

            # Calcular total
            total_venta = precio_prod * cantidad
            print(f"Total a pagar: S/ {total_venta:.2f}")

            confirmar = input("¿Confirmar venta? (s/n): ").strip().lower()
            if confirmar == 's':
                # Actualizar stock en la tabla productos
                nuevo_stock = stock_actual - cantidad
                cursor.execute("UPDATE productos SET stock = ? WHERE codigo = ?", (nuevo_stock, codigo_prod))

                # Registrar la venta en la tabla ventas
                from datetime import date
                fecha_actual = date.today().strftime("%Y-%m-%d")
                cursor.execute("INSERT INTO ventas (fecha, total) VALUES (?, ?)", (fecha_actual, total_venta))

                conexion.commit()
                guardar_cambios()
                print("\n¡Venta realizada con exito y stock actualizado!")
            else:
                print("Venta cancelada.")

        elif opcion == "2":
            conexion.close()
            print("Regresando al menu principal...")
            break
        else:
            print("Opcion no valida.")

        conexion.close()




# ============================================================
# MODULO 3: MONITOREO DE ALERTAS
# RESPONSABLE: INTEGRANTE 3
# ============================================================


# ============================================================
# MODULO 3: MONITOREO DE ALERTAS DE STOCK
# RESPONSABLE: INTEGRANTE 3
# ============================================================

def modulo_alertas():

    while True:

        print("\n========================================")
        print("       MONITOREO DE ALERTAS DE STOCK")
        print("========================================")
        print("[1] Consultar estado de todos los productos")
        print("[2] Mostrar productos con stock bajo")
        print("[3] Buscar alertas de un producto")
        print("[4] Volver al menu principal")
        print("========================================")

        opcion = input("Ingrese una opcion (1-4): ")

        # Conectar con la base de datos
        conexion = sqlite3.connect(ruta_db)
        cursor = conexion.cursor()

        # OPCION 1: Consultar todos los productos
        if opcion == "1":

            cursor.execute("""
                SELECT codigo, nombre, stock, stock_minimo
                FROM productos
                ORDER BY nombre
            """)

            productos = cursor.fetchall()

            print("\n--- ESTADO GENERAL DEL STOCK ---")

            if len(productos) == 0:
                print("No hay productos registrados.")

            else:
                alertas = 0

                for producto in productos:

                    codigo = producto[0]
                    nombre = producto[1]
                    stock = producto[2]
                    stock_minimo = producto[3]

                    print("\n----------------------------------------")
                    print("Codigo:", codigo)
                    print("Producto:", nombre)
                    print("Stock actual:", stock)
                    print("Stock minimo:", stock_minimo)

                    if stock < stock_minimo:
                        print("Estado: ALERTA DE STOCK BAJO")
                        alertas += 1
                    else:
                        print("Estado: STOCK SUFICIENTE")

                print("\n========================================")
                print("RESUMEN DEL MONITOREO")
                print("Total de productos:", len(productos))
                print("Productos con stock bajo:", alertas)
                print("Productos con stock suficiente:",
                      len(productos) - alertas)
                print("========================================")

        # OPCION 2: Mostrar productos con stock bajo
        elif opcion == "2":

            cursor.execute("""
                SELECT codigo, nombre, stock, stock_minimo
                FROM productos
                WHERE stock < stock_minimo
                ORDER BY stock ASC
            """)

            productos = cursor.fetchall()

            print("\n--- PRODUCTOS CON STOCK BAJO ---")

            if len(productos) == 0:
                print("No existen productos con stock bajo.")

            else:
                for producto in productos:
                    print("\n----------------------------------------")
                    print("Codigo:", producto[0])
                    print("Producto:", producto[1])
                    print("Stock actual:", producto[2])
                    print("Stock minimo:", producto[3])
                    print("ALERTA: Se necesita reponer stock.")

                print("\nTotal de alertas:", len(productos))

        # OPCION 3: Buscar alertas de un producto
        elif opcion == "3":

            codigo = input(
                "Ingrese el codigo del producto (ejemplo P003): "
            ).strip()

            cursor.execute("""
                SELECT codigo, nombre, stock, stock_minimo
                FROM productos
                WHERE codigo = ?
            """, (codigo,))

            producto = cursor.fetchone()

            if producto is None:
                print("\nProducto no encontrado.")

            else:
                codigo = producto[0]
                nombre = producto[1]
                stock = producto[2]
                stock_minimo = producto[3]

                print("\n--- RESULTADO DE LA CONSULTA ---")
                print("Codigo:", codigo)
                print("Producto:", nombre)
                print("Stock actual:", stock)
                print("Stock minimo:", stock_minimo)

                if stock < stock_minimo:
                    print("ALERTA: STOCK BAJO")
                    print("Se necesita reponer este producto.")
                else:
                    print("Estado: STOCK SUFICIENTE")

        # OPCION 4: Volver al menu principal
        elif opcion == "4":
            conexion.close()
            print("\nRegresando al menu principal...")
            break

        else:
            print("\nOpcion no valida. Intente nuevamente.")

        conexion.close()
        input("\nPresione Enter para continuar...")

# ============================================================
# MODULO 4: MODULO DE REPORTES
# RESPONSABLE: INTEGRANTE 4
# ============================================================

def reporte_ventas_por_dia(conexion):
    print("\n--- REPORTE DE VENTAS POR DÍA ---")
    try:
        consulta = "SELECT fecha, total FROM ventas"
        df_ventas = pd.read_sql_query(consulta, conexion)

        if df_ventas.empty:
            print("No hay ventas registradas para analizar.")
            return

        # Análisis complejo usando Pandas: agrupar por fecha y sumar totales
        ventas_diarias = df_ventas.groupby('fecha')['total'].sum().reset_index()
        ventas_diarias.columns = ['Fecha', 'Total de Ingresos (S/)']

        # Imprimir el DataFrame estructurado
        print(ventas_diarias.to_string(index=False))
        print(f"\nTotal acumulado histórico: S/ {ventas_diarias['Total de Ingresos (S/)'].sum():.2f}")

    except Exception as e:
        print(f"Error al generar el reporte: {e}\n(Verifique que la tabla 'ventas' exista y tenga datos)")

def reporte_estado_inventario(conexion):
    print("\n--- ESTADO CONSOLIDADO DEL INVENTARIO ---")
    consulta = "SELECT codigo, nombre, precio, stock, stock_minimo FROM productos"
    df_inventario = pd.read_sql_query(consulta, conexion)

    if df_inventario.empty:
        print("El inventario está vacío.")
        return

    # Renombrar columnas para mejor presentación en consola
    df_inventario.columns = ['Código', 'Producto', 'Precio (S/)', 'Stock Actual', 'Stock Mínimo']

    # Calcular el valor total almacenado por producto (Precio * Stock) usando Pandas
    df_inventario['Valor Total (S/)'] = df_inventario['Precio (S/)'] * df_inventario['Stock Actual']

    print(df_inventario.to_string(index=False))
    print("-" * 50)
    print(f"Valor total de toda la mercancía en almacén: S/ {df_inventario['Valor Total (S/)'].sum():.2f}")

def modulo_reportes():

    conexion = sqlite3.connect(ruta_db)

    while True:
        print("\n========================================")
        print("          MODULO DE REPORTES")
        print("========================================")
        print("  [1] Análisis de ventas por día")
        print("  [2] Estado consolidado del inventario")
        print("  [3] Volver al menú principal")
        print("========================================")

        opcion = input("Ingrese una opción (1-3): ")

        if opcion == "1":
            reporte_ventas_por_dia(conexion)
        elif opcion == "2":
            reporte_estado_inventario(conexion)
        elif opcion == "3":
            print("Regresando al menú principal...")
            break
        else:
            print("Opción no válida.")

    conexion.close()

# ============================================================
# MENU PRINCIPAL DEL SISTEMA
# ============================================================

def mostrar_menu():

    print("\n============================================================")
    print("      SISTEMA DE MONITOREO DE INVENTARIO Y FACTURACION")
    print("============================================================")

    print("Seleccione un modulo para comenzar:\n")

    print("  [1] Modulo de Inventario (Registro de Productos)")
    print("  [2] Modulo de Facturacion (Nueva Venta)")
    print("  [3] Monitoreo de Alertas de Stock")
    print("  [4] Modulo de Reportes")
    print("  [5] Salir del Sistema")

    print("============================================================")


def main():

    while True:

        mostrar_menu()

        opcion = input("Ingrese una opcion (1-5): ")

        if opcion == "1":
            modulo_inventario()

        elif opcion == "2":
            modulo_facturacion()

        elif opcion == "3":
            modulo_alertas()

        elif opcion == "4":
            modulo_reportes()

        elif opcion == "5":
            print("\nCerrando el sistema... Hasta luego!")
            break

        else:
            print("\nError: Opcion no valida.")


# ============================================================
# INICIO DEL SISTEMA
# ============================================================

if __name__ == "__main__":
    main()



      SISTEMA DE MONITOREO DE INVENTARIO Y FACTURACION
Seleccione un modulo para comenzar:

  [1] Modulo de Inventario (Registro de Productos)
  [2] Modulo de Facturacion (Nueva Venta)
  [3] Monitoreo de Alertas de Stock
  [4] Modulo de Reportes
  [5] Salir del Sistema
Ingrese una opcion (1-5): 4

          MODULO DE REPORTES
  [1] Análisis de ventas por día
  [2] Estado consolidado del inventario
  [3] Volver al menú principal
Ingrese una opción (1-3): 2

--- ESTADO CONSOLIDADO DEL INVENTARIO ---
Código           Producto  Precio (S/)  Stock Actual  Stock Mínimo  Valor Total (S/)
  P001   Teclado Logitech         85.0            18             5            1530.0
  P002        Mouse Gamer         45.0            15             5             675.0
  P003    Monitor Samsung        650.0             4             5            2600.0
  P004     Audifonos Sony        120.0            12             4            1440.0
  P005   Memoria USB 32GB         35.0            30            10     

In [ ]:
import sqlite3

try:
    conexion.rollback()
    conexion.close()
    print("Conexion anterior cerrada.")
except (NameError, sqlite3.Error):
    print("No se encontro una conexion abierta.")

No se encontro una conexion abierta.
